# Impact of Dried Blood Spot Diameter and Transit Time on Analyte Concentrations and Cut-Off Classification in Routine Newborn Screening

## Effect of DBS diameter on results in paired DBS

This Jupyter notebook describes the analysis of duplicate measurements of IRT and TSH in the same analytical batch, part of the following manuscript (submitted for publication in Clinical Chemistry):

{Citation placeholder - Once accepted for publication, a citation will appear here}

Please cite this paper if you re-use any of the code in this analysis.

### Library imports

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
import statsmodels as sm

In [2]:
sns.set_style("whitegrid")

## Load plate and well information on repeats from Panthera

We have extracted plate and and well information from the Panthera using the piexif library. This is saved in the file blood_spot_pseudonymisation.csv. Load this data:

In [3]:
df = pd.read_csv('data/blood_spot_pseudonymisation.csv')
df["date_time"] = pd.to_datetime(df["date_time"], errors="coerce")

In [4]:
df.head()

,repeats_pseudonym,Episode_pseudo,plate,well,date_time
0,REPEATS201504_00001,EPI1504705,EXMSMSp1,B7,2015-04-01 12:56:42
1,REPEATS201504_00002,EPI1504705,063315440456,C7,2015-04-01 12:56:48
2,REPEATS201504_00003,EPI1504705,063315440538,H8,2015-04-02 14:11:23
3,REPEATS201504_00004,EPI1504705,063315440538,H9,2015-04-02 14:11:28
4,REPEATS201504_00005,EPI1504729,EXMSMSp1,D7,2015-04-01 13:03:35


## Load DBS metrics

Load DBS metrics, obtained using methods described in the following paper.

    Flynn N, Moat SJ, Hogg SL. A computer vision approach to the assessment of dried blood spot size and quality in newborn screening. Clin Chim Acta. 2023;547:117418.

Data is stored by Panthera instrument and month, so we load each each month separately before combining into a single dataframe. 

Note this is for all images obtained from the Panthera. We will merge with just the samples we need later.

In [5]:
year = (2015,2016,2017,2018,2019,2020,2021,2022,2023)
month = (1,2,3,4,5,6,7,8,9,10,11,12)
dataframes = []

for y in year:
    for m in month:
        try:
            filepath = 'data/dbs_metrics/' + str(y) + '-' + str(m) + '.csv'
            df_month = pd.read_csv(filepath,parse_dates=['month','date_time'])
            dataframes.append(df_month)
        except:
            print('No data for year ' + str(y) + ' and month ' + str(m))

No data for year 2015 and month 1
No data for year 2023 and month 11
No data for year 2023 and month 12


In [ ]:
year = (2023,2024,2025)
month = (1,2,3,4,5,6,7,8,9,10,11,12)

for y in year:
    for m in month:
        try:
            filepath = 'data/dbs_metrics/' + str(y) + '-' + str(m) + '_new.csv'
            df_month = pd.read_csv(filepath,parse_dates=['month','date_time'])
            dataframes.append(df_month)
        except:
            print('No data for year ' + str(y) + ' and month ' + str(m))

No data for year 2023 and month 1
No data for year 2023 and month 2
No data for year 2023 and month 3
No data for year 2023 and month 4
No data for year 2023 and month 5
No data for year 2023 and month 6
No data for year 2023 and month 7
No data for year 2023 and month 8
No data for year 2023 and month 9


In [ ]:
dbs_metrics = pd.concat(dataframes,ignore_index=True)

In [ ]:
len(dbs_metrics)

Now we merge with Panthera plate and well information, keeping only episodes that were punched and analysed on the same day.

In [ ]:
#Make sure both dates are stored as date_time
df['date_time'] = pd.to_datetime(df['date_time'])
dbs_metrics['date_time'] = pd.to_datetime(dbs_metrics['date_time'])

In [ ]:
df = pd.merge(
    df,
    dbs_metrics,
    on=["Episode_pseudo", "date_time"],
    how="inner"  # only keep matching rows
)

In [ ]:
len(df)

## Load GSP data

Load data including plate information from GSP

In [ ]:
def report_counts(tsh, irt):
    print("TSH: " + str(len(tsh)))
    print("IRT: " + str(len(irt)))    

In [ ]:
GSP_TSH = pd.read_csv('data/gspcht_0621-1125.csv',parse_dates=['RunDate'])
GSP_IRT = pd.read_csv('data/gspcf_0621-1125.csv',parse_dates=['RunDate'])

report_counts(tsh=GSP_TSH, irt=GSP_IRT)

In [ ]:
GSP_TSH.head()

In [ ]:
GSP_TSH.tail()

Drop any non-numeric concentrations

In [ ]:
GSP_TSH = GSP_TSH[~GSP_TSH['Conc'].isna()]
GSP_IRT = GSP_IRT[~GSP_IRT['Conc'].isna()]

report_counts(tsh=GSP_TSH, irt=GSP_IRT)

GSP IRT has lost leading zeros on plate code due to earlier file manipulation. Fix this by adding in the leading zero if the plate code is only 11 characters (expect this to be 12)

In [ ]:
GSP_IRT['PlateCode'] = GSP_IRT['PlateCode'].astype(str)
GSP_IRT.loc[GSP_IRT['PlateCode'].str.len() == 11, 'PlateCode'] = \
    '0' + GSP_IRT.loc[GSP_IRT['PlateCode'].str.len() == 11, 'PlateCode']

In [ ]:
GSP_IRT['PlateCode']

Current GSP was in routine use from 1st July 2021. Drop results before July 2021 (these are method verification results rather than routine use)

In [ ]:
GSP_TSH = GSP_TSH[GSP_TSH['RunDate'] >= "2021-07-01"]
GSP_IRT = GSP_IRT[GSP_IRT['RunDate'] >= "2021-07-01"]

report_counts(tsh=GSP_TSH, irt=GSP_IRT)

## Load Omnilab results (pre latest GSP)

Results were obtained via SQL query from the Omnilab LIMS system. Note this does not include plate information.

In [ ]:
Omni_results = pd.read_csv('data/repeat_in_duplicate_0415-0925.csv',parse_dates=["Date Rec'd"])

In [ ]:
len(Omni_results)

Drop results after July 2021 - we can use GSP data for these and pair with plate information

In [ ]:
Omni_results = Omni_results[Omni_results["Date Rec'd"]<"2021-07-01"]

In [ ]:
len(Omni_results)

In [ ]:
Omni_results.head()

Create a new column to identify which test the result is for (identified via Test code name)

In [ ]:
Omni_results = Omni_results.copy()
Omni_results['Test'] = Omni_results['Test Code'].str.extract(r'(^[A-Za-z]+)')

Count the number of tests for each analyte

In [ ]:
Omni_results['Test'].value_counts()

There is not enough data to assess relationship for amino acids or acylcarnitines (we also have a problem for IMDs where plates are not named uniquely), so create separate dataframes for IRT and TSH only.

In [ ]:
OMNI_IRT = Omni_results[Omni_results['Test'] == 'IRT'].copy()
OMNI_TSH = Omni_results[Omni_results['Test'] == 'TSH'].copy()


report_counts(tsh=OMNI_TSH, irt=OMNI_IRT)

Create placeholder values for results outside measurable range

In [ ]:
# Convert values that start with '>' to 251
OMNI_TSH.loc[OMNI_TSH['Result'].astype(str).str.startswith('>'), 'Result'] = 251

# Convert values that start with '<' to 1.2
OMNI_TSH.loc[OMNI_TSH['Result'].astype(str).str.startswith('<'), 'Result'] = 1.2

report_counts(tsh=OMNI_TSH, irt=OMNI_IRT)

Check for non-numeric results

In [ ]:
OMNI_TSH[pd.to_numeric(OMNI_TSH['Result'], errors='coerce').isna()]

Note :INS and :NT are codes to indicate that the sample was not tested, so we can safely exclude these.

In [ ]:
OMNI_IRT[pd.to_numeric(OMNI_IRT['Result'], errors='coerce').isna()]

Coerce Result to float - we'll lose the non-numeric results above. 

In [ ]:
OMNI_TSH['Result'] = pd.to_numeric(OMNI_TSH['Result'], errors='coerce')
OMNI_IRT['Result'] = pd.to_numeric(OMNI_IRT['Result'], errors='coerce')

report_counts(tsh=OMNI_TSH, irt=OMNI_IRT)

Now we drop rows in Omni_results where the sample already appears in the GSP dataframes (this should have already been done via the date check above, but this is a second check).

In [ ]:
OMNI_TSH = OMNI_TSH[~OMNI_TSH['Episode_pseudo'].isin(GSP_TSH['Episode_pseudo'])]
OMNI_IRT = OMNI_IRT[~OMNI_IRT['Episode_pseudo'].isin(GSP_IRT['Episode_pseudo'])]

report_counts(tsh=OMNI_TSH, irt=OMNI_IRT)

## Pre-processing

After we have merged the spot metrics with plate information, the main dataframe contains many columns. We will only include a subset that we are interested in.

In [ ]:
cols_to_include = ['Episode_pseudo','plate','well','date_time','number_punches','equiv_diam_mm','pred_multi','prob_multi']
df = df[cols_to_include]

Exclude any repeats that at least 7 days after the first appearance of the lab no. Routine NBS would be expected to be completed by the then, and any additional analyses are likely for quality assurance (e.g. reagent acceptance testing, method verifications)

In [ ]:
# find first date for each lab_no
first_dates = df.groupby('Episode_pseudo')['date_time'].transform('min')

# filter rows within 7 days of the first appearance
df = df[df['date_time'] <= first_dates + pd.Timedelta(days=7)]

Add a leading zero for single digits wells, so that all wells have the same format (e.g. A7 to A07) and they can be accurately matched.

In [ ]:
# Add leading zero for single-digit wells
df['well'] = df['well'].str.replace(r'([A-H])(\d)$', r'\g<1>0\g<2>', regex=True)

In [ ]:
#Extract date for datetime
df['date'] = df['date_time'].dt.date

Keep only rows where (lab_no, plate) combination appears more than once (i.e. the episode appears more than once on the same plate)

In [ ]:
df_filtered = df.groupby(['Episode_pseudo', 'plate']).filter(lambda x: len(x) > 1)

## Merge on plate positions

Here we use data from the GSP instrument to merge results with DBS metrics, where the laboratory number, plate and well match.

In [ ]:
## TSH
GSP_TSH['PlateCode'] = GSP_TSH['PlateCode'].astype(str)
df_filtered['plate'] = df_filtered['plate'].astype(str)

TSH_merged_on_plate_positions = GSP_TSH.merge(df_filtered,
                                              left_on=['Episode_pseudo','PlateCode','Well'],
                                              right_on=['Episode_pseudo','plate','well'],
                                              how='inner')
TSH_merged_on_plate_positions['merge_type'] = 'plate_positions'


#IRT
GSP_IRT['PlateCode'] = GSP_IRT['PlateCode'].astype(str)
df_filtered['plate'] = df_filtered['plate'].astype(str)

IRT_merged_on_plate_positions = GSP_IRT.merge(df_filtered,
                                              left_on=['Episode_pseudo','PlateCode','Well'],
                                              right_on=['Episode_pseudo','plate','well'],
                                              how='inner')
IRT_merged_on_plate_positions['merge_type'] = 'plate_positions'

report_counts(tsh=TSH_merged_on_plate_positions,irt=IRT_merged_on_plate_positions)

Drop duplicate rows where episode, plate and well position are the same, because we can't unambigously match punch and results.

In [ ]:
TSH_merged_on_plate_positions = TSH_merged_on_plate_positions[~TSH_merged_on_plate_positions.duplicated(subset=['Episode_pseudo', 'plate','well'], keep=False)]
IRT_merged_on_plate_positions = IRT_merged_on_plate_positions[~IRT_merged_on_plate_positions.duplicated(subset=['Episode_pseudo', 'plate','well'], keep=False)]

report_counts(tsh=TSH_merged_on_plate_positions,irt=IRT_merged_on_plate_positions)

Check the sample quality via incorrect application classifier for each dried blood spot. 0304 category is incorrect blood application.

In [ ]:
TSH_merged_on_plate_positions['pred_multi'].value_counts()

In [ ]:
IRT_merged_on_plate_positions['pred_multi'].value_counts()

Exclude incorrect blood application DBS

In [ ]:
TSH_merged_on_plate_positions = TSH_merged_on_plate_positions[TSH_merged_on_plate_positions['pred_multi'] == 'controls']
IRT_merged_on_plate_positions = IRT_merged_on_plate_positions[IRT_merged_on_plate_positions['pred_multi'] == 'controls']

report_counts(tsh=TSH_merged_on_plate_positions,irt=IRT_merged_on_plate_positions)

Only keep rows where the episode number appears twice in the dataset (i.e. two results for each specimen)

In [ ]:
TSH_merged_on_plate_positions = TSH_merged_on_plate_positions.groupby('Episode_pseudo').filter(lambda x: len(x) == 2)
IRT_merged_on_plate_positions = IRT_merged_on_plate_positions.groupby('Episode_pseudo').filter(lambda x: len(x) == 2)

report_counts(tsh=TSH_merged_on_plate_positions,irt=IRT_merged_on_plate_positions)

In [ ]:
cols = ['Episode_pseudo','Conc','equiv_diam_mm','pred_multi','prob_multi','merge_type']

In [ ]:
TSH_merged_on_plate_positions[cols].head()

## Merge with Episode number and result order

Before 8th May 2018, TSH was only recorded to integer values in the LIMs so will exclude these results, as rounding will likely dilute the observed effect.

In [ ]:
OMNI_TSH = OMNI_TSH[OMNI_TSH["Date Rec'd"] > "2018-05-07"]

Include only panthera images that appear twice.

In [ ]:
df_two = df_filtered[df_filtered['Episode_pseudo'].map(df_filtered['Episode_pseudo'].value_counts()) == 2]

Add an occurence index for each episode in both the omnilab results and Panthera images. The assumption here is that results are uploaded to Omnilab in the same order in which they are punched. We will later assess the impact of this assumption in a sensitivity analysis.

In [ ]:
# Add an occurrence index for each Episode/lab_no
OMNI_TSH['occurrence'] = OMNI_TSH.groupby('Episode_pseudo').cumcount()
OMNI_IRT['occurrence'] = OMNI_IRT.groupby('Episode_pseudo').cumcount()
df_two['occurrence'] = df_two.groupby('Episode_pseudo').cumcount()

report_counts(tsh=OMNI_TSH,irt=OMNI_IRT)

Now merge on both the sample episode number and occurence index

In [ ]:
# TSH
TSH_merged_on_result_order = pd.merge(
    OMNI_TSH,
    df_two,
    left_on=['Episode_pseudo', 'occurrence'],
    right_on=['Episode_pseudo', 'occurrence'],
    how='inner'
)
TSH_merged_on_result_order['merge_type'] = 'result_order'


# IRT
IRT_merged_on_result_order = pd.merge(
    OMNI_IRT,
    df_two,
    left_on=['Episode_pseudo', 'occurrence'],
    right_on=['Episode_pseudo', 'occurrence'],
    how='inner'
)
IRT_merged_on_result_order['merge_type'] = 'result_order'

report_counts(tsh=TSH_merged_on_result_order,irt=IRT_merged_on_result_order)

Drop any incorrect application samples, as we did earlier for GSP results

In [ ]:
TSH_merged_on_result_order['pred_multi'].value_counts()

In [ ]:
IRT_merged_on_result_order['pred_multi'].value_counts()

In [ ]:
TSH_merged_on_result_order = TSH_merged_on_result_order[TSH_merged_on_result_order['pred_multi'] == 'controls']
IRT_merged_on_result_order = IRT_merged_on_result_order[IRT_merged_on_result_order['pred_multi'] == 'controls']

report_counts(tsh=TSH_merged_on_result_order,irt=IRT_merged_on_result_order)

Keep episodes where the episode appears twice

In [ ]:
TSH_merged_on_result_order = TSH_merged_on_result_order.groupby('Episode_pseudo').filter(lambda x: len(x) == 2)
IRT_merged_on_result_order = IRT_merged_on_result_order.groupby('Episode_pseudo').filter(lambda x: len(x) == 2)

report_counts(tsh=TSH_merged_on_result_order,irt=IRT_merged_on_result_order)

Rename column so we can concatenate with GSP Data, as the results columns currently have different names

In [ ]:
TSH_merged_on_result_order = TSH_merged_on_result_order.rename(columns={'Result': 'Conc'})
IRT_merged_on_result_order = IRT_merged_on_result_order.rename(columns={'Result': 'Conc'})

report_counts(tsh=TSH_merged_on_result_order,irt=IRT_merged_on_result_order)

## Combine dataframes

Now we combine the data obtained from merging on plate positions, and those merged on result order.

In [ ]:
tsh = pd.concat([TSH_merged_on_plate_positions[cols],TSH_merged_on_result_order[cols]], ignore_index=True)
irt = pd.concat([IRT_merged_on_plate_positions[cols],IRT_merged_on_result_order[cols]], ignore_index=True)

report_counts(tsh=tsh,irt=irt)

We now load data on repeats from Omnilab - these are samples where we are expecting to have repeats. (Note the count is lower as this is test sets, which contain two results, whereas elsewhere it is the number of results)

In [ ]:
repeats = pd.read_csv('data/repeat_in_duplicate_0415-0925.csv')

## TSH
tsh_repeats = repeats[(repeats['Test Set'] == 'N010A') | (repeats['Test Set'] == 'N010B')]
tsh_repeats = tsh_repeats[['Episode_pseudo','Test Set']].drop_duplicates()

## IRT
irt_repeats = repeats[(repeats['Test Set'] == 'N025A') | (repeats['Test Set'] == 'N031')]
irt_repeats = irt_repeats[['Episode_pseudo','Test Set']].drop_duplicates()

report_counts(tsh=tsh_repeats,irt=irt_repeats)

Keep only TSH and IRT repeats that we are expecting.

In [ ]:
tshc = tsh[tsh['Episode_pseudo'].isin(tsh_repeats['Episode_pseudo'])]
irtc = irt[irt['Episode_pseudo'].isin(irt_repeats['Episode_pseudo'])]

report_counts(tsh=tshc,irt=irtc)

In [ ]:
# Only keep results where there are two episodes
tsh = tsh.groupby('Episode_pseudo').filter(lambda x: len(x) == 2)
irt = irt.groupby('Episode_pseudo').filter(lambda x: len(x) == 2)

report_counts(tsh=tsh,irt=irt)

For each DBS, calculate averages and differences

In [ ]:
## TSH
tsh['avg_value'] = tsh.groupby('Episode_pseudo')['Conc'].transform('mean')
tsh['avg_equiv_diam_mm'] = tsh.groupby('Episode_pseudo')['equiv_diam_mm'].transform('mean')

# Find smallest and largest DBS
tsh['smallest_dbs'] = tsh.groupby('Episode_pseudo')['equiv_diam_mm'].transform('min')
tsh['largest_dbs'] = tsh.groupby('Episode_pseudo')['equiv_diam_mm'].transform('max')

#x2 as make it a difference
tsh['diff_value'] = 2*(tsh['Conc'] - tsh['avg_value'])
tsh['diff_equiv_diam_mm'] = 2*(tsh['equiv_diam_mm'] - tsh['avg_equiv_diam_mm'])

# Compute percentage differences
tsh['pct_diff_value'] = (tsh['diff_value'] / tsh['avg_value']) * 100

## IRT
irt['avg_value'] = irt.groupby('Episode_pseudo')['Conc'].transform('mean')
irt['avg_equiv_diam_mm'] = irt.groupby('Episode_pseudo')['equiv_diam_mm'].transform('mean')

# Find smallest and largest DBS
irt['smallest_dbs'] = irt.groupby('Episode_pseudo')['equiv_diam_mm'].transform('min')
irt['largest_dbs'] = irt.groupby('Episode_pseudo')['equiv_diam_mm'].transform('max')

#x2 as make it a difference
irt['diff_value'] = 2*(irt['Conc'] - irt['avg_value'])
irt['diff_equiv_diam_mm'] = 2*(irt['equiv_diam_mm'] - irt['avg_equiv_diam_mm'])

# Compute percentage differences
irt['pct_diff_value'] = (irt['diff_value'] / irt['avg_value']) * 100

report_counts(tsh=tsh,irt=irt)

Exclude episodes if either pair or results is outside measurable range

In [ ]:
# Exclude anything outside reportable range
tsh = tsh[tsh['Conc'] >= 1.3]
tsh = tsh[tsh['Conc'] < 250]

irt = irt[irt['Conc'] >= 9]
irt = irt[irt['Conc'] < 500]

# Only keep results where there are two episodes
tsh = tsh.groupby('Episode_pseudo').filter(lambda x: len(x) == 2)
irt = irt.groupby('Episode_pseudo').filter(lambda x: len(x) == 2)

report_counts(tsh=tsh,irt=irt)

Keep only the larger spot

In [ ]:
tsh = tsh[tsh['diff_equiv_diam_mm'] > 0]
irt = irt[irt['diff_equiv_diam_mm'] > 0]

report_counts(tsh=tsh,irt=irt)

## Dataset descriptive statistics

In [ ]:
tsh['avg_equiv_diam_mm'].describe()

In [ ]:
irt['avg_equiv_diam_mm'].describe()

In [ ]:
tsh['diff_equiv_diam_mm'].describe()

In [ ]:
irt['diff_equiv_diam_mm'].describe()

In [ ]:
tsh['avg_value'].describe()

In [ ]:
irt['avg_value'].describe()

In [ ]:
tsh['smallest_dbs'].min()

In [ ]:
irt['smallest_dbs'].min()

In [ ]:
tsh['largest_dbs'].max()

In [ ]:
irt['largest_dbs'].max()

## Supplemental Figure

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(8,8), dpi=600)

sns.set_theme(style="whitegrid")

# ----- Custom x-axis labels -----
xlabel_tsh_value = "Average TSH concentration (IU/L)"
xlabel_irt_value = "Average IRT concentration (ng/mL)"

# ----- PANEL A: TSH – avg_equiv_diam_mm -----
bin_width = 0.5
bins_a = np.arange(min(tsh["avg_equiv_diam_mm"].dropna()), max(tsh["avg_equiv_diam_mm"].dropna()) + bin_width, bin_width)


axes[0, 0].hist(tsh["avg_equiv_diam_mm"].dropna(), bins=bins_a, color='black')
axes[0, 0].set_xlabel("Average DBS diameter (mm): TSH")
axes[0, 0].set_ylabel("Count")
axes[0, 0].set_title("A", loc='left', fontsize=14, fontweight='bold')
axes[0, 0].set_xlim(4,16)
axes[0, 0].set_ylabel("")

# ----- PANEL B: IRT – avg_equiv_diam_mm -----

bin_width = 0.5
bins_b = np.arange(min(irt["avg_equiv_diam_mm"].dropna()), max(irt["avg_equiv_diam_mm"].dropna()) + bin_width, bin_width)

axes[0, 1].hist(irt["avg_equiv_diam_mm"].dropna(), bins=bins_b, color='black')
axes[0, 1].set_xlabel("Average DBS diameter (mm): IRT")
axes[0, 1].set_ylabel("Count")
axes[0, 1].set_title("B", loc='left', fontsize=14, fontweight='bold')
axes[0, 1].set_xlim(4,16)
axes[0, 1].set_ylabel("")


# ----- PANEL C: TSH – diff_equiv_diam_mm -----
bin_width = 0.2
bins_c = np.arange(min(tsh["diff_equiv_diam_mm"].dropna()), max(tsh["diff_equiv_diam_mm"].dropna()) + bin_width, bin_width)

axes[1, 0].hist(tsh["diff_equiv_diam_mm"].dropna(), bins=bins_c, color='black')
axes[1, 0].set_xlabel("Difference in DBS diameter (mm): TSH")
axes[1, 0].set_ylabel("Count")
axes[1, 0].set_title("C", loc='left', fontsize=14, fontweight='bold')
axes[1, 0].set_xlim(0,6)
axes[1, 0].set_ylabel("")

# ----- PANEL D: IRT – diff_equiv_diam_mm -----
bin_width = 0.2
bins_d = np.arange(min(irt["diff_equiv_diam_mm"].dropna()), max(irt["diff_equiv_diam_mm"].dropna()) + bin_width, bin_width)

axes[1, 1].hist(irt["diff_equiv_diam_mm"].dropna(), bins=bins_d, color='black')
axes[1, 1].set_xlabel("Difference in DBS diameter (mm): IRT")
axes[1, 1].set_ylabel("Count")
axes[1, 1].set_title("D", loc='left', fontsize=14, fontweight='bold')
axes[1, 1].set_xlim(0,6)
axes[1, 1].set_ylabel("")

## ----- PANEL E: TSH – concentration

# Bin width
bin_width = 2

# Regular bins from 0 to 50
bin_edges = np.arange(0, 52, bin_width)   # 0,2,4,...,50

overflow_bin = 51  # bucket all >50 here

# Cap values above 50
tsh['capped_value'] = np.where(
    tsh['avg_value'] > overflow_bin, overflow_bin, tsh['avg_value']
)

ax = axes[2,0]  # target subplot

# Plot histogram with custom bins
sns.histplot(
    tsh['capped_value'],
    bins=np.append(bin_edges, overflow_bin + 1),  # include overflow bin
    color='black',
    alpha=1,
    edgecolor='white',
    linewidth=1,
    ax=ax
)

# Desired ticks
ticks = [0, 10, 20, 30, 40, 50]   # 51 = overflow position
tick_labels = ["0", "10", "20", "30", "40", ">50"]

ax.set_xticks(ticks)
ax.set_xticklabels(tick_labels)
ax.set_ylabel("")
ax.set_xlabel(xlabel_tsh_value)
ax.set_xlim(0, 52)
axes[2, 0].set_title("E", loc='left', fontsize=14, fontweight='bold')

## ----- PANEL F: IRT – concentration

# Bin width
bin_width = 5

# Regular bins from 0 to 140
bin_edges = np.arange(0, 142, bin_width)   # 0,2,4,...,50

overflow_bin = 145  # bucket all >50 here

# Cap values above 140
irt['capped_value'] = np.where(
    irt['avg_value'] > overflow_bin, overflow_bin, irt['avg_value']
)

ax = axes[2,1]  # target subplot

# Plot histogram with custom bins
sns.histplot(
    irt['capped_value'],
    bins=np.append(bin_edges, overflow_bin + 1),  # include overflow bin
    color='black',
    alpha=1,
    edgecolor='white',
    linewidth=1,
    ax=ax
)

# Desired ticks
ticks = [0, 20, 40, 60, 80, 100, 120, 141]   # 151 = overflow position
tick_labels = ["0", "20", "40", "60", "80", "100", "120", ">140"]

ax.set_xticks(ticks)
ax.set_xticklabels(tick_labels)
ax.set_ylabel("")
ax.set_xlim(0, 145)
axes[2, 1].set_xlabel(xlabel_irt_value)
axes[2, 1].set_title("F", loc='left', fontsize=14, fontweight='bold')

plt.tight_layout()

Add categories for difference to help visualise differences

In [ ]:
tsh['average_mm_diff_category'] = pd.cut(tsh['diff_equiv_diam_mm'],
                                               bins=[0,0.5,1,100],
                                              labels=["0-0.5","0.5-1.0",">1"])

irt['average_mm_diff_category'] = pd.cut(irt['diff_equiv_diam_mm'],
                                               bins=[0,0.5,1,100],
                                              labels=["0-0.5","0.5-1.0",">1"])

In [ ]:
tsh['average_mm_diff_category'].value_counts()

In [ ]:
irt['average_mm_diff_category'].value_counts()

## OLS analysis (full dataset)

In [ ]:
import statsmodels.api as sm

X = sm.add_constant(tsh['diff_equiv_diam_mm'])
y = tsh['pct_diff_value']

model = sm.OLS(y, X).fit()
print(model.summary())

In [ ]:
X = sm.add_constant(irt['diff_equiv_diam_mm'])
y = irt['pct_diff_value']

model = sm.OLS(y, X).fit()
print(model.summary())

## Define subgroups and sensitivity analyses

In [ ]:
## TSH

# Paired on well positions
tsh_pp = tsh[tsh['merge_type'] == 'plate_positions']

# DBS diameter difference < 1.5 mm
tsh_dbsd1 = tsh[tsh['diff_equiv_diam_mm'] <= 1.5]

# TSH 6.0 – 10.0 mU/
tsh610 = tsh[(tsh['avg_value'] >= 6.0) & (tsh['avg_value'] < 10)]

# TSH ≥ 8.0 mU/L
tsh8 = tsh[tsh['avg_value'] >= 8.0]

# DBS diameter tertiles
tsh['dbs_tertile'], bins = pd.qcut(tsh['smallest_dbs'], q=3, labels=['low', 'mid', 'high'], retbins=True)

tsh_tertile1 = tsh[tsh['dbs_tertile'] == 'low']
tsh_tertile2 = tsh[tsh['dbs_tertile'] == 'mid']
tsh_tertile3 = tsh[tsh['dbs_tertile'] == 'high']


dfs_tsh={
    "All": tsh,
    "Paired on well positions":tsh_pp,
"DBS diameter difference < 1.5 mm":tsh_dbsd1,
"TSH 6.0 – 10.0 mU/L":tsh610,
"TSH ≥ 8.0 mU/L":tsh8,
f"Lower DBS diameter tertile (< {round(bins[1],1)} mm)":tsh_tertile1,
f"Central DBS diameter tertile ({round(bins[1],1)} - {round(bins[2],1)} mm)":tsh_tertile2,
f"Upper DBS diameter tertile (≥ {round(bins[2],1)} mm)":tsh_tertile3
}


## IRT
# Paired on well positions
irt_pp = irt[irt['merge_type'] == 'plate_positions']

# DBS diameter difference < 1.5 mm
irt_dbsd1 = irt[irt['diff_equiv_diam_mm'] <= 1.5]

# IRT 42.0 - 70.0 ng/mL
irt25 = irt[(irt['avg_value'] >= 42) & (irt['avg_value'] <= 70)]

# IRT ≥ 56 ng/mL
irt56 = irt[irt['avg_value']>=56]

## DBS diameter tertiles
irt['dbs_tertile'], bins = pd.qcut(irt['smallest_dbs'], q=3, labels=['low', 'mid', 'high'], retbins=True)

irt_tertile1 = irt[irt['dbs_tertile'] == 'low']
irt_tertile2 = irt[irt['dbs_tertile'] == 'mid']
irt_tertile3 = irt[irt['dbs_tertile'] == 'high']

dfs_irt={
    "All": irt,
    "Paired on well positions":irt_pp,
"DBS diameter difference < 1.5 mm":irt_dbsd1,
"IRT 42.0 – 70.0 ng/mL":irt25,
"IRT ≥ 56 ng/mL":irt56,
f"Lower DBS diameter tertile (< {round(bins[1],1)} mm)":irt_tertile1,
f"Central DBS diameter tertile ({round(bins[1],1)} - {round(bins[2],1)} mm)":irt_tertile2,
f"Upper DBS diameter tertile (≥ {round(bins[2],1)} mm)":irt_tertile3
}


In [ ]:
def run_ols_across_dataframes(df_dict, x_var='diff_equiv_diam_mm', y_var='pct_diff_value'):
    """
    Function to produce sensitivity/subgroup analysis table
    
    df_dict: dict of {name: dataframe}
    x_var: predictor variable
    y_var: outcome variable
    """

    results = []

    for name, df in df_dict.items():
        d = df[[x_var, y_var]].dropna()
        n_samples = len(d)

        X = sm.add_constant(d[x_var])
        y = d[y_var]

        model = sm.OLS(y, X).fit()

        coef = model.params[x_var]
        ci_low, ci_high = model.conf_int().loc[x_var]
        p_value = model.pvalues[x_var]
        r2 = model.rsquared

        # Format coefficient + CI (2 decimal places)
        coef_with_ci = f"{coef:.2f} ({ci_low:.2f} - {ci_high:.2f})"

        # Format p-value
        if p_value < 0.001:
            p_val_formatted = "< 0.001"
        else:
            p_val_formatted = f"{p_value:.3f}"

        results.append({
            "Subgroup": name,
            "n": n_samples,
            "% change in analyte concentration per mm change in DBS diameter difference (95% confidence interval)": coef_with_ci,
            "R²": round(r2, 3),
            "p-value": p_val_formatted
        })

    return pd.DataFrame(results)


In [ ]:
tsh_ols_results = run_ols_across_dataframes(dfs_tsh)
tsh_ols_results.to_csv('tsh_duplicates_ols.csv',index=False)
tsh_ols_results

In [ ]:
irt_ols_results = run_ols_across_dataframes(dfs_irt)
irt_ols_results.to_csv('irt_duplicates_ols.csv',index=False)
irt_ols_results

## Figure 

In [ ]:
def plot_mean_ci_figure(df, analyte, ax=None, panel_label=None):

    # --- Compute mean and 95% CI per category ---
    summary = (
        df.groupby('average_mm_diff_category', observed=True)['pct_diff_value']
          .agg(['mean', 'count', 'std'])
          .assign(
              sem=lambda x: x['std'] / np.sqrt(x['count']),
              ci95=lambda x: 1.96 * x['std'] / np.sqrt(x['count']),
              lower=lambda x: x['mean'] - 1.96 * x['std'] / np.sqrt(x['count']),
              upper=lambda x: x['mean'] + 1.96 * x['std'] / np.sqrt(x['count'])
          )
          .reset_index()
          .dropna(subset=['mean'])
    )

    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 6), dpi=600)

    # Plot mean points
    sns.pointplot(
        data=summary,
        x='average_mm_diff_category',
        y='mean',
        color='black',
        join=False,
        errorbar=None,
        ax=ax
    )

    # Add 95% CI error bars
    ax.errorbar(
        x=np.arange(len(summary)),
        y=summary['mean'],
        yerr=summary['ci95'],
        fmt='none',
        ecolor='black',
        capsize=4,
        alpha=1
    )

    # Labels
    ax.set_xlabel("")
    #ax.set_xlabel('Difference in DBS diameter (mm)')
    ax.set_ylabel(f"Mean {analyte} % difference")
    
    if panel_label is not None:
        ax.text(
            0.05, 0.95, panel_label,
            transform=ax.transAxes,
            fontsize=14,
            va='top',
            ha='left',
            bbox=dict(
            facecolor='white',
            edgecolor='black',
            boxstyle='round,pad=0.5'
        )
        )
    
    return ax

In [ ]:
dfs = [tsh, irt, tsh610, irt25]
analytes = ['TSH', 'IRT', 'TSH', 'IRT']
figure_labels = ["TSH (all)", "IRT (all)", "TSH 6-10 mU/L", "IRT 42-70 ng/mL"]
panel_labels = ["A","B","C","D"]

fig, axes = plt.subplots(2, 2, figsize=(10, 10), dpi=600)

for ax, df, analyte, label, figure_label in zip(axes.flat, dfs, analytes, panel_labels, figure_labels):
    plot_mean_ci_figure(df, analyte, ax=ax, panel_label=figure_label)
    ax.set_title(label, loc='left', fontsize=16, fontweight='bold')

for ax in axes.flat:
    ax.set_ylim(-2, 10)
    
# Shared x label (bottom centre)
fig.text(0.52, 0.02, "Difference in DBS diameter (mm)",
         ha="center", va="center", fontsize=18)

plt.tight_layout(rect=[0, 0.02, 1, 1])
plt.savefig('Figure1.jpg')

plt.show()

Get counts for figure legend

In [ ]:
def format_category_legend(series):
    
    counts = series.value_counts()
    total_n = counts.sum()
    parts = []
    
    for cat, n in counts.items():
        if cat == ">1":
            label = ">1.0 mm"
        else:
            label = f"{cat} mm"
        parts.append(f"{label}: n={n}")
    
    return f"n={total_n}, Numbers per category: " + "; ".join(parts)


In [ ]:
for df in dfs:
    print(format_category_legend(df['average_mm_diff_category']))